<h1 style=\"text-align: center; font-size: 50px;\"> <h1 style=\"text-align: center; font-size: 50px;\"> 📦 Register Model </h1> </h1>

This notebook packages the **audio-native agentic workflow** as an **MLflow pyfunc model**, logs it with artifacts
(index, config), and registers it to the MLflow Model Registry for serving.

- Retrieval: **CLAP** audio↔text embeddings over timestamped audio windows (+ **MMR** reranker)
- Generation: **Qwen Omni** listens to the selected audio windows and answers (no transcripts required)
- Orchestration: **LangGraph** (relevance → memory → retrieve → rerank → answer → memoize)
- Vector store: **FAISS** (in-model artifact or built on first run)
- Memory: disk-backed key-value cache (per-corpus+question)


# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- KV Memory
- LLM Setup
- State Model
- Node Functions
- Graph Definition
- Graph Visualization
- Generated Answer
- Message History

# Start Execution

In [1]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations
from pathlib import Path  # Object-oriented file system paths

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import (  # Utility functions for logging, LLM I/O, and schema generation
    load_config,
    load_secrets,
    load_secrets_to_env,
    get_project_root,
    logger,
    setup_model_environment,
)

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Install and Import Libraries

In [3]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 34.3 ms, sys: 12.5 ms, total: 46.8 ms
Wall time: 1.89 s


In [4]:
from __future__ import annotations  # Enables postponed evaluation of annotations (PEP 563)

# ─────── Standard Library ───────
import json  # JSON serialization and deserialization
import warnings  # Issue warning messages
from collections import namedtuple  # Factory for creating tuple subclasses with named fields
from datetime import datetime  # Date and time utilities
from pathlib import Path  # Object-oriented filesystem paths
from typing import Any, Dict, List, Literal, Optional, TypedDict  # Type hinting support
import numpy as np  # Numerical operations and array handling
import soundfile as sf  # Reading and writing sound files

# ─────── Third-Party Packages ───────
import mlflow  # Model tracking and serving framework
import mlflow.pyfunc  # MLflow Python function interface for custom models
from mlflow.tracking import MlflowClient  # Interface to interact with MLflow tracking server for experiments, runs, and artifacts
from IPython.display import Markdown, display  # IPython utilities for notebook output formatting
from tqdm import tqdm  # Visual progress bar for loops
import torch # PyTorch for tensor computations and deep learning
import torchaudio # Audio processing library built on PyTorch
import faiss # Library for efficient similarity search and clustering of dense vectors
import pandas as pd # Data manipulation and analysis

# Qwen Omni (audio+video+text) – both full & Thinker-only variants
from transformers import Qwen2_5OmniProcessor, Qwen2_5OmniThinkerForConditionalGeneration # Qwen Omni processor and model
from transformers import AutoProcessor as ClapProcessor, ClapModel # CLAP processor and model for audio embeddings
from qwen_omni_utils import process_mm_info     # official utils to prep audio/video inputs

# ─────── LangChain Core & Community ───────
from langgraph.graph import StateGraph, END

# ─────── Local application-specific imports ───────
from src.agentic_workflow import build_agentic_graph # Custom workflow builder for agentic tasks
from src.agentic_audio_rag_model import AgenticAudioRAGModel  # Custom model for audio RAG tasks
from src.model_selection import ModelSelector
from src.simple_kv_memory import SimpleKVMemory  # In-memory key-value store for agent state
from src.simple_kv_memory import _mem_get, _mem_put  # Functions for getting and putting items in memory
from src.generate_test_audio import generate_test_audio, generate_and_convert_formats  # Functions to generate test audio files
from src.segment_audio_embeddings import (  # Functions for segmenting audio and extracting embeddings
    clap_embed_audio,
    clap_embed_text,
    segment_audio_embeddings, 
    rerank_hits_mmr, 
    retrieve_and_rerank,
    ensure_wav
)


# Configure Settings

In [5]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [6]:
project_root = get_project_root()
DOCS: list
FILE_ID: str
INPUT_PATH: Path = Path("../data/input")  

MEMORY_PATH: Path = Path("../data/memory")
CONFIG_PATH = "../configs/config.yaml"
SECRETS_PATH = "../configs/secrets.yaml"

EXPERIMENT_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Experiment"
RUN_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Run"
MODEL_NAME = "AIStudio-Agentic-Audio-RAG-with-LangGraph-Model"

# --- Retrieval / Rerank params (must match run-workflow) ---
MEMORY_FILENAME = "kv_memory.json"
INDEX_VECS_NPY = "audio_vecs.npy"
INDEX_META_JSON = "audio_meta.json"

# --- Retrieval defaults (keep in sync with run-workflow notebook) ---
RELEVANCE_THRESHOLD = 0.18
FETCH_K = 24            # breadth for stage-1
TOP_K   = 6             # final segments
MEMOIZE_MIN_SCORE = 0.5 # cache only if score is above this threshold

# --- Audio and embedding models ---
AUDIO_LLM = "Qwen/Qwen2.5-Omni-7B"
CLAP_ID = "laion/clap-htsat-unfused"
FORCE_CPU = bool(int(os.environ.get("AIS_FORCE_CPU", "0")))

# --- Media extensions (must match the workflow) ---
AUDIO_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a"}
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
MEDIA_EXTS = AUDIO_EXTS | VIDEO_EXTS

SRC_DIR   = project_root / "src"
DATA_DIR = project_root / "data" / "input"
ARTIF_DIR = project_root / "artifacts"

# Make src importable
sys.path.insert(0, str(SRC_DIR))

# logger.info("Project root:", project_root)
logger.info("Src dir    : %s", SRC_DIR)
logger.info("Inputs     : %s", DATA_DIR)
logger.info("Artifacts  : %s", ARTIF_DIR)

## Configuration and Secrets Loading

In this section, we load configuration parameters and API keys from separate YAML files. This separation helps maintain security by keeping sensitive information (API keys) separate from configuration settings.

- **config.yaml**: Contains non-sensitive configuration parameters like model sources and URLs
- **secrets.yaml**: Contains sensitive API keys for services like HuggingFace
- *(Optional for Premium users)* Secrets such as API keys for services like HuggingFace can be stored as environment variables for the project and loaded into the notebook (see the project's README file for steps on how to save secrets in Secrets Manager).

In [7]:
# Load secrets from secrets.yaml file (if it exists) into environment
if Path(SECRETS_PATH).exists():
    load_secrets_to_env(SECRETS_PATH)
else:
    print(f"No secrets file found at {SECRETS_PATH}; relying on preexisting environment")

# Retrieve secrets from environment
try:
    secrets = load_secrets()
except ValueError:
    secrets = {}

# Load configuration and secrets
config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")
print("✅ Secrets loaded successfully")

✅ Loaded 1 secrets into environment variables.
✅ Configuration loaded successfully
✅ Secrets loaded successfully


# Verify Assets

In [8]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

def log_secrets_status(secrets: Dict[str, Any], success_message: str, failure_message: str) -> None:
    """
    Logs the status of secrets based on their existence.

    Parameters:
        secrets (Dict[str, Any]): Secrets retrieved to check if they exist.
        success_message (str): Message to log if secrets exists.
        failure_message (str): Message to log if secrets do not exist.
    """
    if secrets:
        logger.info(f"Project secrets are available. {success_message}")
    else:
        logger.info(f"There are no project secrets found. {failure_message}")

In [9]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)

log_asset_status(
    asset_path=CONFIG_PATH,
    asset_name="Config",
)

log_secrets_status(
    secrets=secrets,
    success_message="",
    failure_message="Please check if the secrets were propely connfigured in your secrets yaml file or in Secrets Manager."
)

# KV Memory

In [10]:
# memory: SimpleKVMemory = SimpleKVMemory(MEMORY_PATH)
# memory.set('dummy key', 'dummy value')

# Prepare Audio Files for Inference

In [11]:
# logger.info("🎧 Scanning directory for media files: %s", INPUT_PATH)

# Ensure HF cache into project area
setup_model_environment()

# Initialize CLAP on CPU to avoid VRAM pressure during packaging
clap_processor = ClapProcessor.from_pretrained(CLAP_ID)
clap_model = ClapModel.from_pretrained(CLAP_ID).eval()
try:
    clap_model.to("cpu")
    torch.cuda.empty_cache()
    logger.info("CLAP moved to CPU; GPU cache cleared.")
except Exception as e:
    logger.warning("CLAP offload skipped: %s", e)

# Ensure artifacts folder layout
(ARTIF_DIR / "index").mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "config").mkdir(parents=True, exist_ok=True)
(ARTIF_DIR / "memory").mkdir(parents=True, exist_ok=True)


# Audio Indexing and Embeddings

In [12]:
# Build a fresh CLAP index from INPUT files (same logic as the workflow, but persisted)
logger.info("📂 Scanning & indexing media from: %s", str(DATA_DIR))

audio_index, media_paths = segment_audio_embeddings(
    clap_processor=clap_processor,
    clap_model=clap_model,
    INPUT_PATH=str(DATA_DIR),
    MEDIA_EXTS=MEDIA_EXTS,
    AUDIO_EXTS=AUDIO_EXTS,
    VIDEO_EXTS=VIDEO_EXTS,
)

# Persist raw vectors by re-embedding each saved window (same order as meta)
# (FAISS can't be reliably pickled across runtimes; vectors+meta is the robust route)
vecs = []
for m in audio_index.meta:
    wav_path = ensure_wav(AUDIO_EXTS, VIDEO_EXTS, m["file_path"])
    audio, sr = sf.read(wav_path)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    i0 = max(0, int(m["start_s"] * sr))
    i1 = max(i0, int(m["end_s"]   * sr))
    seg = audio[i0:i1].astype(np.float32, copy=False)
    v = clap_embed_audio(clap_processor, clap_model, seg, sr)
    v = v / (np.linalg.norm(v) + 1e-12)
    vecs.append(v)

if vecs:
    vecs = np.stack(vecs, axis=0).astype(np.float32)
else:
    vecs = np.zeros((0, 512), dtype=np.float32)

# Save artifacts
np.save(ARTIF_DIR / "index" / INDEX_VECS_NPY, vecs)
with open(ARTIF_DIR / "index" / INDEX_META_JSON, "w") as f:
    json.dump(audio_index.meta, f, indent=2)

# Write runtime config
cfg = {
    "relevance_threshold": RELEVANCE_THRESHOLD,
    "fetch_k": FETCH_K,
    "top_k": TOP_K,
    "clap_repo": CLAP_ID,
    "media_root": str(DATA_DIR),
    "memoize_min_score": MEMOIZE_MIN_SCORE,
}
with open(ARTIF_DIR / "config" / "config.json", "w") as f:
    json.dump(cfg, f, indent=2)

logger.info("📇 Indexed %d segments across %d files", len(audio_index.meta), len(media_paths))


# MLflow Registration

In [13]:
from typing import Annotated
import operator

Messages = Annotated[List[Dict[str, Any]], operator.add]

class AudioState(TypedDict, total=False):
    question: str
    file_id: str
    index: Any
    memory: Any
    audio_llm: Any

    is_relevant: bool
    from_memory: bool
    hits_raw: List[Dict[str, Any]]
    hits: List[Dict[str, Any]]
    evidence: List[Dict[str, Any]]
    answer: str

    messages: Messages
    

In [ ]:
class AudioAgenticPyFunc(mlflow.pyfunc.PythonModel):
    """
    MLflow pyfunc for Audio-only RAG with:
      - CLAP (CPU) for retrieval + MMR reranking
      - Qwen2.5 Omni Thinker for audio reasoning
      - LangGraph for memory → retrieve → generate → memoize
    """

    # ---------- helpers ----------
    @staticmethod
    def _mem_get(mem, key):
        return mem.get(key) if hasattr(mem, "get") else None

    @staticmethod
    def _mem_put(mem, key, value):
        if hasattr(mem, "set"):
            mem.set(key, value)

    @staticmethod
    def _build_faiss(vecs: np.ndarray) -> faiss.IndexFlatIP:
        idx = faiss.IndexFlatIP(vecs.shape[1])
        # vectors already normalized above; re-normalize defensively
        _v = vecs / (np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12)
        idx.add(_v.astype(np.float32))
        return idx

    # ---------- load_context ----------
    def load_context(self, context):
        setup_model_environment()

        # Artifacts
        index_dir   = Path(context.artifacts["index_dir"])
        config_path = Path(context.artifacts["config_path"])
        memory_dir  = Path(context.artifacts["memory_dir"])

        vecs_name = INDEX_VECS_NPY
        meta_name = INDEX_META_JSON

        self.vecs  = np.load(index_dir / vecs_name).astype(np.float32)
        with open(index_dir / meta_name, "r") as f:
            self.metas = json.load(f)
        with open(config_path, "r") as f:
            cfg = json.load(f)

        self.relevance_threshold = float(cfg.get("relevance_threshold", 0.18))
        self.fetch_k             = int(cfg.get("fetch_k", 24))
        self.top_k               = int(cfg.get("top_k", 6))
        self.memoize_min_score   = float(cfg.get("memoize_min_score", 0.50))
        self.media_root          = cfg.get("media_root")
        self.clap_repo           = cfg.get("clap_repo", CLAP_ID)

        # FAISS (rebuild at runtime)
        self.faiss_index = self._build_faiss(self.vecs)

        # CLAP (CPU)
        self.clap_processor = ClapProcessor.from_pretrained(self.clap_repo)
        self.clap_model     = ClapModel.from_pretrained(self.clap_repo).eval()
        try:
            self.clap_model.to("cpu")
        except Exception:
            pass

        # Memory
        memory_dir.mkdir(parents=True, exist_ok=True)
        self.memory = SimpleKVMemory(Path(memory_dir / MEMORY_FILENAME))

        selector  = ModelSelector()
        local_dir = Path(selector.format_model_path(AUDIO_LLM))
        local_dir.mkdir(parents=True, exist_ok=True)

        device_map = "cpu" if FORCE_CPU or (not torch.cuda.is_available()) else "auto"
        dtype = torch.float16 if (torch.cuda.is_available() and device_map != "cpu") else torch.float32

        self.q_processor = Qwen2_5OmniProcessor.from_pretrained(str(local_dir), trust_remote_code=True)
        self.q_model = Qwen2_5OmniThinkerForConditionalGeneration.from_pretrained(
            str(local_dir), 
            torch_dtype=dtype, 
            device_map=device_map, 
            trust_remote_code=True, 
            low_cpu_mem_usage=True
        ).eval()

        # --- Qwen Adapter (audio-only; decode new tokens only; sanitize role tags) ---
        class _QwenAdapter:
            def __init__(self, proc, model):
                self.processor = proc
                self.model = model

            def _sanitize(self, txt: str) -> str:
                import re
                txt = re.sub(r"^(?:\d+\s*)?(?:Human:|User:|Assistant:|System:)\s*", "", txt, flags=re.IGNORECASE).strip()
                txt = re.sub(r"([!?.])\1{2,}", r"\1", txt)  # collapse repeats
                return txt

            def answer(self, question: str, audio_hits: list, return_audio: bool = False) -> dict:
                import numpy as np, soundfile as sf
                from qwen_omni_utils import process_mm_info

                qwen_default_system = (
                    "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, "
                    "capable of perceiving auditory and visual inputs, as well as generating text and speech."
                )

                user_content = [{
                    "type": "text",
                    "text": ("Answer ONLY using the provided audio clips. "
                            "If the clips do not contain the answer, reply exactly: NOT_FOUND_IN_AUDIO.\n"
                            f"Question: {question}")
                }]

                usable = 0
                for h in (audio_hits or []):
                    audio_full, sr = sf.read(h["wav_path"])
                    if audio_full.ndim == 2:
                        audio_full = audio_full.mean(axis=1)
                    s0, s1 = int(h["start_s"] * sr), int(h["end_s"] * sr)
                    s0 = max(0, min(s0, len(audio_full)))
                    s1 = max(0, min(s1, len(audio_full)))
                    if s1 <= s0:
                        continue
                    seg = audio_full[s0:s1].astype(np.float32)
                    user_content.append({"type": "audio", "audio": seg, "sampling_rate": sr})
                    usable += 1

                if usable == 0:
                    return {"answer": "Not found in audio.", "evidence": []}

                conv = [
                    {"role": "system", "content": [{"type": "text", "text": qwen_default_system}]},
                    {"role": "user",   "content": user_content},
                ]

                text = self.processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
                audios, images, videos = process_mm_info(conv, use_audio_in_video=False)

                audios = audios if (audios and len(audios) > 0) else None
                images = images if (images and len(images) > 0) else None
                videos = videos if (videos and len(videos) > 0) else None

                inputs = self.processor(
                    text=text, audio=audios, images=images, videos=videos,
                    return_tensors="pt", padding=True, use_audio_in_video=False
                ).to(self.model.device)

                tok = getattr(self.processor, "tokenizer", None)
                eos_id = getattr(tok, "eos_token_id", None)
                try:
                    chat_eos = tok.convert_tokens_to_ids("<|im_end|>") if tok is not None else None
                except Exception:
                    chat_eos = None
                eos_ids = [i for i in (eos_id, chat_eos) if i is not None] or None
                pad_id = getattr(tok, "pad_token_id", eos_id)

                with torch.no_grad():
                    out = self.model.generate(
                        **inputs,
                        use_audio_in_video=False,
                        max_new_tokens=192,
                        do_sample=False,
                        num_beams=1,
                        repetition_penalty=1.1,
                        no_repeat_ngram_size=4,
                        eos_token_id=eos_ids,
                        pad_token_id=pad_id,
                        return_dict_in_generate=True,
                    )

                prompt_ids = inputs.get("input_ids", None)
                prompt_len = prompt_ids.shape[1] if prompt_ids is not None else 0
                seqs = getattr(out, "sequences", None)
                if seqs is None:
                    answer = ""
                else:
                    try:
                        new_tokens = seqs[:, prompt_len:]
                    except Exception:
                        new_tokens = seqs
                    decoded = self.processor.batch_decode(new_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=True)
                    answer = (decoded[0].strip() if decoded else "").strip()

                answer = self._sanitize(answer)
                if answer.upper().startswith("NOT_FOUND_IN_AUDIO"):
                    answer = "Not found in audio."

                evid = [{
                    "file_name": h["file_name"], "file_path": h["file_path"],
                    "start_s": h["start_s"], "end_s": h["end_s"],
                    "score": h.get("score_mmr", h.get("score", 0.0)),
                } for h in (audio_hits or [])]

                return {"answer": (answer if answer else "Not found in audio."), "evidence": evid}



        self.audio_llm = _QwenAdapter(self.q_processor, self.q_model)

        # ---- LangGraph (audio-only): memory -> retrieve -> rerank -> generate -> memoize
     #   AudioState = dict

        def node_ingest_question(state: AudioState) -> AudioState:
            q = (state.get("question") or "").strip()
            msgs = state.get("messages", [])
            msgs.append({"role": "developer", "content": "Ingested question"})
            # normalize once for downstream
            return {"question": q, "messages": msgs}

        def node_check_relevance_audio(state: AudioState) -> AudioState:
            # tiny search-only fetch (no MMR) to estimate relevance
            q = state["question"]
            # reuse the same helper used in your workflow: high-recall fetch
            # here we do a very small fetch to keep it cheap
            qvec = clap_embed_text(self.clap_processor, self.clap_model, q)
            qvec = qvec / (np.linalg.norm(qvec) + 1e-12)
            D, I = self.faiss_index.search(qvec[np.newaxis, :].astype(np.float32), 8)
            prelim = []
            for idx, score in zip(I[0], D[0]):
                if 0 <= idx < len(self.metas):
                    m = dict(self.metas[idx]); m["score"] = float(score); prelim.append(m)
            max_score = max((h["score"] for h in prelim), default=-1.0)
            is_rel = max_score >= self.relevance_threshold
            msgs = state.get("messages", [])
            msgs.append({"role":"developer", "content": f"Relevance: max_score={max_score:.3f} -> {'relevant' if is_rel else 'irrelevant'}"})
            out = {"is_relevant": is_rel, "messages": msgs}
            if is_rel and prelim:
                out["hits_raw"] = prelim  # optional: reuse later
            return out

        def node_check_memory(state: AudioState) -> AudioState:
            q = (state.get("question") or "").strip().lower()
            fid = state.get("file_id") or "global"
            key = f"{fid} :: {q}"
            cached = self._mem_get(self.memory, key)
            if cached:
                return {"from_memory": True, "answer": cached.get("answer",""), "evidence": cached.get("evidence", [])}
            return {"from_memory": False}

        def node_ensure_index(state: AudioState) -> AudioState:
            # index is built in load_context(); treat as a no-op for serving
            return {}

        def node_retrieve(state: AudioState) -> AudioState:
            if state.get("from_memory") or not state.get("is_relevant"):
                return {}
            # full fetch + MMR rerank happens across retrieve+rerank split; here just fetch (or reuse prelim)
            hits_raw = state.get("hits_raw")
            if not hits_raw:
                # larger fetch for recall
                q = state["question"]
                qvec = clap_embed_text(self.clap_processor, self.clap_model, q)
                qvec = qvec / (np.linalg.norm(qvec) + 1e-12)
                D, I = self.faiss_index.search(qvec[np.newaxis, :].astype(np.float32), max(self.fetch_k, self.top_k))
                hits_raw = []
                for idx, score in zip(I[0], D[0]):
                    if 0 <= idx < len(self.metas):
                        m = dict(self.metas[idx]); m["score"] = float(score); hits_raw.append(m)
            return {"hits_raw": hits_raw}

        def node_rerank(state: AudioState) -> AudioState:
            if state.get("from_memory") or not state.get("is_relevant"):
                return {}
            hits_raw = state.get("hits_raw", [])
            if not hits_raw:
                return {"hits": []}
            hits = rerank_hits_mmr(
                self.clap_processor, self.clap_model,
                state["question"], hits_raw,
                top_k=self.top_k, fetch_k=self.fetch_k, lam=0.6
            )
            return {"hits": hits}

        def node_generate_audio(state: AudioState) -> AudioState:
            if state.get("from_memory"):
                return {}
            hits = state.get("hits") or []
            if not hits:
                return {"answer": "Not found in audio.", "evidence": []}
            out = self.audio_llm.answer(state["question"], hits, return_audio=False)
            return {"answer": out.get("answer",""), "evidence": out.get("evidence", [])}

        def node_update_memory(state: AudioState) -> AudioState:
            if state.get("from_memory"):
                return {}
            ev = state.get("evidence") or []
            best = max((e.get("score", 0.0) for e in ev), default=0.0)
            if not ev or best < self.memoize_min_score:
                return {}
            q = (state.get("question") or "").strip().lower()
            fid = state.get("file_id") or "global"
            key = f"{fid} :: {q}"
            self._mem_put(self.memory, key, {"answer": state.get("answer",""), "evidence": ev})
            return {}

        def node_output(state: AudioState) -> AudioState:
            # No mutation needed; this mirrors your notebook’s final node.
            return {}

        # Build the graph with the same topology as your workflow:
        g = StateGraph(AudioState)
        g.add_node("ingest_question",       node_ingest_question)
        g.add_node("check_relevance_audio", node_check_relevance_audio)
        g.add_node("check_memory",          node_check_memory)
        g.add_node("ensure_index",          node_ensure_index)
        g.add_node("retrieve",              node_retrieve)
        g.add_node("rerank",                node_rerank)
        g.add_node("generate_audio",        node_generate_audio)
        g.add_node("update_memory",         node_update_memory)
        g.add_node("output_answer",         node_output)

        g.set_entry_point("ingest_question")
        g.add_edge("ingest_question", "check_relevance_audio")

        def after_relevance(state: AudioState):
            return "check_memory" if state.get("is_relevant") else "output_answer"
        g.add_conditional_edges(
            "check_relevance_audio", after_relevance,
            {"check_memory": "check_memory", "output_answer": "output_answer"}
        )

        def after_memory(state: AudioState):
            return "output_answer" if state.get("from_memory") else "ensure_index"
        g.add_conditional_edges(
            "check_memory", after_memory,
            {"output_answer": "output_answer", "ensure_index": "ensure_index"}
        )

        g.add_edge("ensure_index",   "retrieve")
        g.add_edge("retrieve",       "rerank")
        g.add_edge("rerank",         "generate_audio")
        g.add_edge("generate_audio", "update_memory")
        g.add_edge("update_memory",  "output_answer")
        g.add_edge("output_answer",  END)

        self.graph = g.compile()

    # ---------- runtime ----------
    def _invoke(self, question: str, file_id: str = "global") -> dict:
        return self.graph.invoke({
            "question": question, "file_id": file_id,
            "memory": self.memory, "audio_llm": self.audio_llm, "messages": [],
        })

    def predict(self, context, model_input):
        if isinstance(model_input, pd.DataFrame):
            records = model_input.to_dict(orient="records")
        elif isinstance(model_input, list):
            records = model_input
        else:
            raise ValueError("Pass a list of {question, file_id} or a pandas DataFrame.")
        out = []
        for r in records:
            q   = (r.get("question") or "").strip()
            fid = r.get("file_id") or "global"
            s = self._invoke(q, fid)
            out.append({
                "question": q, "file_id": fid,
                "answer": s.get("answer",""),
                "evidence": s.get("evidence", []),
                "from_memory": s.get("from_memory", False),
            })
        return out


/opt/conda/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:168: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [15]:
import mlflow

# mlflow.set_tracking_uri(MLFLOW_TRACKING)
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experiment: {EXPERIMENT_NAME}")

# Read requirements from your repo (or pin a minimal list inline)
req_path = project_root / "requirements.txt"
if req_path.exists():
    with open(req_path, "r") as f:
        pip_reqs = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]
else:
    pip_reqs = [
        "mlflow>=2.10.0",
        "langgraph>=0.2.0",
        "transformers>=4.41.0",
        "torch>=2.1.0",
        "faiss-cpu>=1.7.4",
        "soundfile>=0.12.1",
        "huggingface_hub>=0.23.0",
        "tabulate>=0.9.0",
    ]

with mlflow.start_run(run_name=f"register-{MODEL_NAME}") as run:
    # Artifacts mapping for pyfunc
    artifacts = {
        "index_dir": str(ARTIF_DIR / "index"),
        "config_path": str(ARTIF_DIR / "config" / "config.json"),
        "memory_dir": str(ARTIF_DIR / "memory"),
    }

    # Include local src so the server can import utils, model_selection, etc.
    code_paths = [str(SRC_DIR)]

    model_info = mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=AudioAgenticPyFunc(),
        artifacts=artifacts,
        code_path=code_paths,
        pip_requirements=pip_reqs,
        registered_model_name=MODEL_NAME,
    )

print("Logged model at:", model_info.model_uri)
print("Registered name:", MODEL_NAME)



Using MLflow tracking URI: /phoenix/mlflow
Experiment: AIStudio-Agentic-Audio-RAG-with-LangGraph-Experiment


2025/08/22 19:25:13 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'AIStudio-Agentic-Audio-RAG-with-LangGraph-Model' already exists. Creating a new version of this model...


Logged model at: runs:/2ebca42b580a4e33a562c88565d22298/model
Registered name: AIStudio-Agentic-Audio-RAG-with-LangGraph-Model


Created version '19' of model 'AIStudio-Agentic-Audio-RAG-with-LangGraph-Model'.


In [16]:
loaded = mlflow.pyfunc.load_model(model_info.model_uri)

TEST_Q = "What is the main idea of the content?"
payload = [{"question": TEST_Q, "file_id": "global"}]

res = loaded.predict(payload)
print(json.dumps(res, indent=2)[:1200], "...")


You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.
Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'mrope_section'}


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

[
  {
    "question": "What is the main idea of the content?",
    "file_id": "global",
    "answer": "The!!main!idea!!!of!!!!!the!!content!!is!!!!to!discuss!!!a!bug!!!!!!!!!!!!!!related!to!!!!creating!a!new!project!!!!!!!!!and!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!",
    "evidence": [
      {
        "file_name": "record for bug-20250512_113039-Meeting Recording.mp4",
        "file_path": "/home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data/input/meeting_recording/record for bug-20250512_113039-Meeting Recording.mp4",
        "start_s": 120.0,
        "end_s": 137.856,
        "score": 0.26928157210350034
      },
      {
        "file_name": "record for bug-20250512_113039-Meeting Recording.wav",
        "file_path": "/home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data/input/meeting_recordi

In [17]:
# Ask twice to demonstrate memoization (second call should set from_memory=True if evidence was strong)
print("First call:")
print(loaded.predict([{"question": TEST_Q, "file_id": "global"}]))

print("\nSecond call (cache):")
print(loaded.predict([{"question": TEST_Q, "file_id": "global"}]))


First call:


[{'question': 'What is the main idea of the content?', 'file_id': 'global', 'answer': 'The!! audio!!! audio!! audio!!!! audio!! audio!!!! audio!!! audio!!! audio!!! audio!! audio!!!! audio!!! audio!!!!!! audio!!!!!! audio!!!!!!!! audio!!! audio!!!!!!!! audio!!!!! audio!!!!!!!!! audio!!! audio!!!!!!!! audio!!! audio!! audio!!!! audio!!!!!!!!! audio!!!!!!!!!!!! audio!!! audio!!!!!!!!!!!! audio!! audio! audio!!! audio!!! audio! audio!!!!!!!!!!!! audio!!!!!!!!!!!!! audio! audio!!!!!!!!!!!!!!!!!!! audio!!!!!! audio!!!!!!!!!!!!!!!!!!! audio', 'evidence': [{'file_name': 'record for bug-20250512_113039-Meeting Recording.wav', 'file_path': '/home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data/input/meeting_recording/record for bug-20250512_113039-Meeting Recording.wav', 'start_s': 120.0, 'end_s': 137.856, 'score': 0.26554534435272215}, {'file_name': 'record for bug-20250512_113039-Meeting Recording.mp4', 'file_path': '/home/jovyan/AI-Blueprints/generative-ai/agentic-au

[{'question': 'What is the main idea of the content?', 'file_id': 'global', 'answer': 'The!! audio!!! audio! audio!!!! audio!!!! audio!! audio!!!!! audio!!!!!! audio!!!!!! audio!!! audio!!!!!! audio!!!!!! audio!!!!!!!! audio!!! audio!!!!!!! audio!!! audio!!!!!!!! audio! audio!!! audio!!!!! audio!!!!!!!!!!! audio!!!!!!!!!!!!! audio!!! audio!!!!!!!!!!!! audio!!!!!! audio!!! audio! audio!!! audio!!!!!! audio!!!!! audio!!!!!!!!!!!!! audio!!!!!!!!!!!!! audio!!!! audio!!!!!!!!!!!!! audio!!!!!!!!!!!!! audio!!!!! audio! audio! audio!!! audio!', 'evidence': [{'file_name': 'record for bug-20250512_113039-Meeting Recording.mp4', 'file_path': '/home/jovyan/AI-Blueprints/generative-ai/agentic-audio-rag-with-langgraph/data/input/meeting_recording/record for bug-20250512_113039-Meeting Recording.mp4', 'start_s': 120.0, 'end_s': 137.856, 'score': 0.2724470257759094}, {'file_name': 'record for bug-20250512_113039-Meeting Recording.wav', 'file_path': '/home/jovyan/AI-Blueprints/generative-ai/agentic-aud

In [18]:
# # 1. Set MLflow tracking URI and experiment
# mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "/phoenix/mlflow"))
# mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)
# print(f"Using MLflow tracking URI: {mlflow.get_tracking_uri()}")
# print(f"Experiment: {EXPERIMENT_NAME}")

In [19]:
# %%time

# # These should point to the actual files you're using for model and memory
# MODEL_ARTIFACTS = {
#     "model_path": str(MODEL_PATH),
#     "memory_path": str(MEMORY_PATH),
# }
 
# # === Start MLflow run, log, and register ===
# with mlflow.start_run(run_name=RUN_NAME) as run:
#     print(f"🚀 Started MLflow run: {run.info.run_id}")

#     # Log and register the model using the classmethod
#     AgenticFeedbackModel.log_model(
#         model_name=MODEL_NAME,
#         model_artifacts=MODEL_ARTIFACTS
#     )

# logger.info(f"✅ Model '{MODEL_NAME}' successfully logged and registered.")

In [20]:
# # 3. Retrieve the latest version from the Model Registry
# client = MlflowClient()
# versions = client.get_latest_versions(MODEL_NAME, stages=["None"])

# if not versions:
#     raise RuntimeError(f"No registered versions found for model '{MODEL_NAME}'.")
    
# latest_version = versions[0].version
# model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_version}")

# logger.info(f"Latest registered version of '{MODEL_NAME}': {latest_version}")
# logger.info(f"Signature: {model_info.signature}")

In [21]:
# %%time

# # 4. Load the model from the Model Registry
# loaded_model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_version}")
# logger.info(f"Successfully loaded model '{MODEL_NAME}' version {latest_version} for inference.")

In [22]:
# # 5. Run a sample inference using the loaded model (Audio RAG)

# from pathlib import Path

# # Collect media files (same extensions you used in the ingestion cell)
# _MEDIA_EXTS = {".mp3", ".wav", ".ogg", ".flac", ".m4a", ".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm"}
# sample_media_paths = [
#     str(p) for p in sorted(Path(INPUT_PATH).rglob("*"))
#     if p.is_file() and p.suffix.lower() in _MEDIA_EXTS
# ]

# if not sample_media_paths:
#     raise FileNotFoundError(f"No audio/video files found in {INPUT_PATH}. "
#                             f"Please add at least one media file to run a sample inference.")

# # The audio RAG model expects {"paths": [...], "query": "..."}
# input_payload = [{
#     "paths": sample_media_paths,
#     "query": QUESTION  # reuse your QUESTION var, or set a literal test query here
# }]

# print("\n=== Running Sample Inference (Audio RAG) ===")
# results = loaded_model.predict(input_payload)       # preserve list-in / list-out contract
# result = results[0] if isinstance(results, list) else results

# print("Answer:\n", result["answer"])
# print("\nSupporting chunks:")
# for c in result.get("chunks", []):
#     print(f"- [{c['start_s']:.1f}–{c['end_s']:.1f}s] {c['text'][:120]}...")



# Generated Answer

In [23]:
# display(Markdown(result.answer))

# Message History

In [24]:
# print(result.messages)

In [25]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

INFO:AIS_logger:⏱️ Total execution time: 3m 32.91s


INFO:AIS_logger:✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).